# Gradient Descent – Vom Prinzip zur Praxis

**Gradient Descent** (Gradientenabstieg) ist der fundamentale Optimierungsalgorithmus des Machine Learning. Er findet das Minimum einer Funktion, indem er iterativ in Richtung des steilsten Abstiegs geht – also entgegen des Gradienten.

In diesem Notebook:
1. **1D Gradient Descent** – einfache quadratische Funktion $f(x) = x^2$
2. **2D Gradient Descent** – $f(x, y) = x^2 + y^2$ mit Kontur-Plot
3. **Lernraten-Experimente** – was passiert bei zu großer / zu kleiner Lernrate?

> **Formel:** $x_{t+1} = x_t - \eta \cdot \nabla f(x_t)$  
> $\eta$ (eta) = Lernrate, $\nabla f$ = Gradient

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections.abc import Callable

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

## 1. Die Algorithmen

In [ ]:
def gradient_descent_1d(
    f: Callable[[float], float],
    df: Callable[[float], float],
    x0: float, lr: float = 0.1,
    epochs: int = 100, tol: float = 1e-6
) -> tuple[float, list]:
    """
    Gradient Descent für 1D-Funktionen.
    Findet das Minimum von f(x).

    Returns:
        (x_min, history) — Minimum und Verlauf der x-Werte
    """
    x = x0
    history = [x]
    for _ in range(epochs):
        grad = df(x)
        x_new = x - lr * grad
        history.append(x_new)
        if abs(x_new - x) < tol:
            x = x_new
            break
        x = x_new
    return x, history


def gradient_descent_2d(
    f: Callable[[np.ndarray], float],
    grad_f: Callable[[np.ndarray], np.ndarray],
    x0: np.ndarray, lr: float = 0.1,
    epochs: int = 100, tol: float = 1e-6
) -> tuple[np.ndarray, list]:
    """
    Gradient Descent für mehrdimensionale Funktionen.

    Args:
        f: Zielfunktion f(x) -> float
        grad_f: Gradient ∇f(x) -> np.ndarray
        x0: Startpunkt
    """
    x = x0.copy()
    history = [x.copy()]
    for _ in range(epochs):
        g = grad_f(x)
        x_new = x - lr * g
        history.append(x_new.copy())
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x = x_new
    return x, history

## 2. Gradient Descent in 1D

Wir minimieren $f(x) = x^2$ mit $f'(x) = 2x$. Das globale Minimum liegt bei $x = 0$.

In [ ]:
def f(x):
    return x**2

def df(x):
    return 2 * x

# Gradient Descent von x0=5.0 aus
x_min, history = gradient_descent_1d(f, df, x0=5.0, lr=0.1)
print(f"Gefundenes Minimum: x = {x_min:.6f} (erwartet: 0)")
print(f"Anzahl Schritte: {len(history) - 1}")
print(f"Letzte 5 x-Werte: {[f'{x:.4f}' for x in history[-5:]]}")

In [ ]:
# Visualisierung: Pfad des Gradient Descent
x_vals = np.linspace(-6, 6, 200)
y_vals = f(x_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linker Plot: Funktionskurve + GD-Pfad
ax1.plot(x_vals, y_vals, 'b-', linewidth=2, label='$f(x) = x^2$')
ax1.scatter(history, [f(x) for x in history], c='red', s=30, zorder=5, label='GD-Schritte')
ax1.plot(history, [f(x) for x in history], 'r--', alpha=0.5, linewidth=1)
ax1.scatter([x_min], [f(x_min)], c='green', s=100, marker='*', zorder=6, label=f'Minimum x={x_min:.4f}')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.set_title('Gradient Descent auf $f(x) = x^2$')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Rechter Plot: Konvergenz (x-Wert über Iterationen)
ax2.plot(range(len(history)), history, 'g-o', markersize=4, linewidth=1.5)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5, label='Ziel: x=0')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('x-Wert')
ax2.set_title('Konvergenzverlauf')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Gradient Descent in 2D

Jetzt minimieren wir $f(x, y) = x^2 + y^2$ mit $\nabla f = (2x, 2y)$. Das Minimum liegt bei $(0, 0)$.

In [ ]:
def f2d(x: np.ndarray) -> float:
    return x[0]**2 + x[1]**2

def grad_f2d(x: np.ndarray) -> np.ndarray:
    return np.array([2*x[0], 2*x[1]])

# Start bei (3, 4)
x0 = np.array([3.0, 4.0])
x_min_2d, history_2d = gradient_descent_2d(f2d, grad_f2d, x0, lr=0.1)

print(f"Startpunkt: ({x0[0]}, {x0[1]})")
print(f"Gefundenes Minimum: ({x_min_2d[0]:.6f}, {x_min_2d[1]:.6f})")
print(f"Anzahl Schritte: {len(history_2d) - 1}")

# History als Arrays
hist_x = np.array([p[0] for p in history_2d])
hist_y = np.array([p[1] for p in history_2d])

In [ ]:
# Kontur-Plot mit GD-Pfad
x_range = np.linspace(-5, 5, 100)
y_range = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = X**2 + Y**2

fig, ax = plt.subplots(figsize=(8, 7))

# Konturlinien
contours = ax.contour(X, Y, Z, levels=20, cmap='viridis', alpha=0.6)
ax.clabel(contours, inline=True, fontsize=8, fmt='%.0f')

# GD-Pfad
ax.plot(hist_x, hist_y, 'r-o', markersize=5, linewidth=2, label='GD-Pfad')
ax.scatter([x0[0]], [x0[1]], c='blue', s=120, marker='o', zorder=5, label=f'Start ({x0[0]}, {x0[1]})')
ax.scatter([x_min_2d[0]], [x_min_2d[1]], c='green', s=150, marker='*', zorder=5, label=f'Minimum ({x_min_2d[0]:.2f}, {x_min_2d[1]:.2f})')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Gradient Descent 2D: $f(x,y) = x^2 + y^2$')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 4. Lernraten-Experiment

Die Lernrate $\eta$ ist der wichtigste Hyperparameter. Was passiert bei verschiedenen Werten?

- **Zu klein** ($\eta = 0.01$): langsame Konvergenz, viele Schritte
- **Optimal** ($\eta = 0.1$): schnelle, stabile Konvergenz
- **Zu groß** ($\eta = 0.9$): Oszillation, springt über das Minimum
- **Viel zu groß** ($\eta = 1.01$): Divergenz!

In [ ]:
learning_rates = [0.01, 0.1, 0.5, 0.9, 1.01]
colors = ['blue', 'green', 'orange', 'red', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr, color in zip(learning_rates, colors):
    _, hist = gradient_descent_1d(f, df, x0=5.0, lr=lr, epochs=50)
    label = f'η = {lr}'
    axes[0].plot(range(len(hist)), hist, color=color, linewidth=2, label=label)
    axes[1].plot(range(len(hist)), [f(x) for x in hist], color=color, linewidth=2, label=label)

axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('x-Wert')
axes[0].set_title('x-Konvergenz bei verschiedenen Lernraten')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Funktionswert-Konvergenz')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Lernraten-Experiment: $f(x) = x^2$, Start bei $x_0 = 5$', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Zusammenfassung

| Konzept | Erklärung |
|---------|-----------|
| **Gradient** | Zeigt in Richtung des steilsten *Anstiegs* |
| **Descent** | Wir gehen in die *entgegengesetzte* Richtung (bergab) |
| **Lernrate $\eta$** | Schrittweite pro Iteration – zu groß → Oszillation, zu klein → langsam |
| **Konvergenz** | Der Algorithmus nähert sich dem Minimum, wenn $\eta$ passend gewählt ist |
| **Abbruchkriterium** | `tol` (Toleranz): stoppt, wenn die Änderung sehr klein wird |

**Praxis-Tipp:** In echten ML-Problemen (z.B. neuronale Netze) verwendet man adaptive Optimierer wie **Adam** oder **RMSprop**, die die Lernrate automatisch anpassen.